In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd 
import numpy as np 

In [3]:
pip install nltk gradio PyPDF2 python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import re 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
import string
import gradio as gr
import PyPDF2
from docx import Document 
import os 


In [6]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
#function to read the resume
def read_file(file_path):
    try :
        file_ext = os.path.splitext(file_path)[1].lower()
        if file_ext == '.txt':
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        elif file_ext == '.pdf':
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                text = ""
                for page in reader.pages:
                    text += page.extract_text() or ""
                return text
        elif file_ext == '.docx':
            doc = Document(file_path)
            text = "\n".join([para.text for para in doc.paragraphs])
            return text
        else:
            return "Error: Unsupported file format. Please upload a .txt, .pdf, or .docx file."
    except Exception as e:
        return f"Error: Unable to read file. {str(e)}"

In [11]:
def ats_resume_checker(resume_file, job_description_file=None, target_industry=""):
    report = {
        "Summary": "",
        "Strengths": [],
        "Areas for Improvement": [],
        "Actionable Recommendations": [],
        "ATS Score": 0
    }
    stop_words = set(stopwords.words('english'))
    ats_score = 0

    resume_text = read_file(resume_file) if resume_file else ""
    if resume_text.startswith("Error"):
        return resume_text
        
    def extract_keywords(text):
        tokens = word_tokenize(text.lower())
        tokens = [t for t in tokens if t not in stop_words and t not in string.punctuation]
        return Counter(tokens).most_common(10)
        
    def check_formatting_issues(text):
        issues = []
        if any(ord(char) > 127 for char in text):
            issues.append("Non-standard characters (e.g., special symbols) may prevent ATS parsing.")
        if '\t\t' in text or '|' in text:
            issues.append("Tables or complex formatting may disrupt ATS readability.")
        if text.strip().startswith('Header') or text.strip().endswith('Footer'):
            issues.append("Headers/footers may cause ATS to misread content.")
        return issues

    def check_sections(text):
        sections = ["contact information", "experience", "education", "skills","projects","certifications"]
        found_sections = []
        missing_sections = []
        for section in sections:
            if section.lower() in text.lower():
                found_sections.append(section)
            else:
                missing_sections.append(section)
        return found_sections, missing_sections

    def check_quantifiable_achievements(text):
        achievements = re.findall(r'\b(increased|reduced|improved|saved|generated|delivered|achieved)\b.*?\d+%', text, re.IGNORECASE)
        return len(achievements) > 0, achievements


    def check_action_verbs(text):
        action_verbs = ["managed", "developed", "led", "implemented", "optimized","created","lead","focused","devised","generated","contributed"]
        return any(verb in text.lower() for verb in action_verbs)
        
    if not resume_text.strip():
        return "Error: Please provide a resume file for analysis."

    job_description = ""
    if job_description_file:
        job_description = read_file(job_description_file)
        if job_description.startswith("Error"):
            return job_description

    resume_keywords = extract_keywords(resume_text)
    keyword_score = 50  # Base score for having some keywords
    if job_description:
        job_keywords = extract_keywords(job_description)
        job_keyword_list = [word for word, _ in job_keywords]
        resume_keyword_list = [word for word, _ in resume_keywords]
        matched_keywords = [word for word in resume_keyword_list if word in job_keyword_list]
        keyword_score += len(matched_keywords) * 5
        if matched_keywords:
            report["Strengths"].append(f"Resume matches {len(matched_keywords)} job keywords: {', '.join(matched_keywords)}.")
        missing_keywords = [word for word in job_keyword_list if word not in resume_keyword_list][:3]
        if missing_keywords:
            report["Areas for Improvement"].append(f"Missing critical keywords: {', '.join(missing_keywords)}.")
            report["Actionable Recommendations"].append(f"1. Add keywords '{', '.join(missing_keywords)}' to Work Experience or Skills.")

    formatting_issues = check_formatting_issues(resume_text)
    if not formatting_issues:
        report["Strengths"].append("Resume uses ATS-compatible formatting (no tables or headers).")
        ats_score += 25
    else:
        report["Areas for Improvement"].extend(formatting_issues)
        report["Actionable Recommendations"].append("2. Remove tables, headers, or special characters; use Arial or Times New Roman.")


    found_sections, missing_sections = check_sections(resume_text)
    if len(found_sections) >= 3:
        report["Strengths"].append(f"Resume includes key sections: {', '.join(found_sections)}.")
        ats_score += 20
    if missing_sections:
        report["Areas for Improvement"].append(f"Missing essential sections: {', '.join(missing_sections)}.")
        report["Actionable Recommendations"].append(f"3. Include '{', '.join(missing_sections[:2])}' sections for ATS compatibility.")


    has_achievements, achievements = check_quantifiable_achievements(resume_text)
    if has_achievements:
        report["Strengths"].append(f"Resume includes measurable results (e.g., '{achievements[0]}').")
        ats_score += 15
    else:
        report["Areas for Improvement"].append("No measurable results found (e.g., 'Increased sales by 20%').")
        report["Actionable Recommendations"].append("4. Add metrics like 'Increased revenue by X%' to Work Experience.")

    if check_action_verbs(resume_text):
        report["Strengths"].append("Resume uses strong action verbs (e.g., 'Managed', 'Developed').")
        ats_score += 10
    else:
        report["Areas for Improvement"].append("No strong action verbs found.")
        report["Actionable Recommendations"].append("5. Start bullet points with verbs like 'Led', 'Developed', or 'Optimized'.")


    if target_industry:
        industry_keywords = {
            "tech": ["software", "programming", "python", "agile", "cloud","Ops","CI/CD","ML","AI"],
            "healthcare": ["patient", "clinical", "medical", "emr", "hipaa"]
        }
        if target_industry.lower() in industry_keywords:
            industry_matches = [kw for kw in industry_keywords[target_industry.lower()] if kw in resume_text.lower()]
            if industry_matches:
                report["Strengths"].append(f"Resume includes {target_industry} terms: {', '.join(industry_matches)}.")
                ats_score += 10
            else:
                report["Areas for Improvement"].append(f"No {target_industry} keywords found.")
                report["Actionable Recommendations"].append(f"6. Add {target_industry} terms like '{', '.join(industry_keywords[target_industry.lower()][:2])}'.")


    ats_score = min(keyword_score + ats_score, 100)
    report["ATS Score"] = ats_score
    report["Summary"] = f"Resume ATS Score: {ats_score}/100. {'Strong but needs minor tweaks.' if ats_score >= 80 else 'Needs targeted improvements for ATS.'}"

    # Step 5: Format Report
    output = "**ATS Resume Analysis Report**\n\n"
    output += f"**Summary**: {report['Summary']}\n\n"
    output += "**Strengths**:\n" + ("\n".join([f"- {s}" for s in report["Strengths"]]) or "- None identified.") + "\n\n"
    output += "**Areas for Improvement**:\n" + ("\n".join([f"- {i}" for i in report["Areas for Improvement"]]) or "- None identified.") + "\n\n"
    output += "**Top Recommendations**:\n" + ("\n".join([r for r in report["Actionable Recommendations"][:5]]) or "- None identified.") + "\n"

    return output

In [14]:
def gradio_interface(resume_file, job_description_file, target_industry):
    return ats_resume_checker(resume_file, job_description_file, target_industry)


interface = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.File(label="Upload Resume File (.txt, .pdf, .docx)", type="filepath"),
        gr.File(label="Upload Job Description File (.txt, .pdf, .docx)", type="filepath"),
        gr.Textbox(placeholder="Enter target industry (e.g., tech, healthcare)", label="Target Industry (Optional)")
    ],
    outputs=gr.Textbox(label="ATS Analysis Report",lines=25),
    title="ATS Resume Checker & Optimizer",
    description="Upload your resume and job description files (.txt, .pdf, or .docx) to analyze ATS compatibility and receive tailored optimization suggestions."
)


if __name__ == "__main__":
    interface.launch()

* Running on local URL:  http://127.0.0.1:7862
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://fd068a5d17c139b28a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error